In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle

In [2]:
df1 = pd.read_csv("train.csv")
df1.head()


,ID,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,...,gender,ethnicity,jaundice,austim,contry_of_res,used_app_before,result,age_desc,relation,Class/ASD
0,1,1,0,1,0,1,0,1,0,1,...,f,?,no,no,Austria,no,6.351166,18 and more,Self,0
1,2,0,0,0,0,0,0,0,0,0,...,m,?,no,no,India,no,2.255185,18 and more,Self,0
2,3,1,1,1,1,1,1,1,1,1,...,m,White-European,no,yes,United States,no,14.851484,18 and more,Self,1
3,4,0,0,0,0,0,0,0,0,0,...,f,?,no,no,United States,no,2.276617,18 and more,Self,0
4,5,0,0,0,0,0,0,0,0,0,...,m,?,no,no,South Africa,no,-4.777286,18 and more,Self,0


In [3]:
df1.columns

Index(['ID', 'A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score',
       'A6_Score', 'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score', 'age',
       'gender', 'ethnicity', 'jaundice', 'austim', 'contry_of_res',
       'used_app_before', 'result', 'age_desc', 'relation', 'Class/ASD'],
      dtype='object')

In [4]:
df1["age"] = df1["age"].astype(int)
df1.head()

,ID,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,...,gender,ethnicity,jaundice,austim,contry_of_res,used_app_before,result,age_desc,relation,Class/ASD
0,1,1,0,1,0,1,0,1,0,1,...,f,?,no,no,Austria,no,6.351166,18 and more,Self,0
1,2,0,0,0,0,0,0,0,0,0,...,m,?,no,no,India,no,2.255185,18 and more,Self,0
2,3,1,1,1,1,1,1,1,1,1,...,m,White-European,no,yes,United States,no,14.851484,18 and more,Self,1
3,4,0,0,0,0,0,0,0,0,0,...,f,?,no,no,United States,no,2.276617,18 and more,Self,0
4,5,0,0,0,0,0,0,0,0,0,...,m,?,no,no,South Africa,no,-4.777286,18 and more,Self,0


In [5]:
df1.isnull().sum()

ID                 0
A1_Score           0
A2_Score           0
A3_Score           0
A4_Score           0
A5_Score           0
A6_Score           0
A7_Score           0
A8_Score           0
A9_Score           0
A10_Score          0
age                0
gender             0
ethnicity          0
jaundice           0
austim             0
contry_of_res      0
used_app_before    0
result             0
age_desc           0
relation           0
Class/ASD          0
dtype: int64

In [6]:
for col in df1.columns:
    num_cols = ["ID", "age", "result"]
    if col not in num_cols:
        print(col, df1[col].unique())
        print("-"*50)

A1_Score [1 0]
--------------------------------------------------
A2_Score [0 1]
--------------------------------------------------
A3_Score [1 0]
--------------------------------------------------
A4_Score [0 1]
--------------------------------------------------
A5_Score [1 0]
--------------------------------------------------
A6_Score [0 1]
--------------------------------------------------
A7_Score [1 0]
--------------------------------------------------
A8_Score [0 1]
--------------------------------------------------
A9_Score [1 0]
--------------------------------------------------
A10_Score [1 0]
--------------------------------------------------
gender ['f' 'm']
--------------------------------------------------
ethnicity ['?' 'White-European' 'Middle Eastern ' 'Pasifika' 'Black' 'Others'
 'Hispanic' 'Asian' 'Turkish' 'South Asian' 'Latino' 'others']
--------------------------------------------------
jaundice ['no' 'yes']
--------------------------------------------------
austim

In [7]:
print(df1['age'])

0      38
1      47
2       7
3      23
4      43
       ..
795    16
796    20
797     5
798    16
799    46
Name: age, Length: 800, dtype: int64


In [8]:
df2 = df1.drop(columns=["ID", "age_desc","relation","ethnicity"])

In [9]:
df2.shape

(800, 18)

In [10]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   A1_Score         800 non-null    int64  
 1   A2_Score         800 non-null    int64  
 2   A3_Score         800 non-null    int64  
 3   A4_Score         800 non-null    int64  
 4   A5_Score         800 non-null    int64  
 5   A6_Score         800 non-null    int64  
 6   A7_Score         800 non-null    int64  
 7   A8_Score         800 non-null    int64  
 8   A9_Score         800 non-null    int64  
 9   A10_Score        800 non-null    int64  
 10  age              800 non-null    int64  
 11  gender           800 non-null    object 
 12  jaundice         800 non-null    object 
 13  austim           800 non-null    object 
 14  contry_of_res    800 non-null    object 
 15  used_app_before  800 non-null    object 
 16  result           800 non-null    float64
 17  Class/ASD       

In [11]:
df2.head()

,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,age,gender,jaundice,austim,contry_of_res,used_app_before,result,Class/ASD
0,1,0,1,0,1,0,1,0,1,1,38,f,no,no,Austria,no,6.351166,0
1,0,0,0,0,0,0,0,0,0,0,47,m,no,no,India,no,2.255185,0
2,1,1,1,1,1,1,1,1,1,1,7,m,no,yes,United States,no,14.851484,1
3,0,0,0,0,0,0,0,0,0,0,23,f,no,no,United States,no,2.276617,0
4,0,0,0,0,0,0,0,0,0,0,43,m,no,no,South Africa,no,-4.777286,0


In [12]:
print(df2['gender'].unique())
df2['gender'] = df2['gender'].replace({
    'f': 1, 'female': 1,
    'm': 0, 'male': 0
})
df2['jaundice'] = df2['jaundice'].replace({
    'yes': 1, 
    'no': 0
})
df2['austim'] = df2['austim'].replace({
    'yes': 1, 
    'no': 0
})
df2['used_app_before'] = df2['used_app_before'].replace({
    'yes': 1, 
    'no': 0
})


['f' 'm']


C:\Users\nehar\AppData\Local\Temp\ipykernel_60220\3405042881.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df2['gender'] = df2['gender'].replace({
C:\Users\nehar\AppData\Local\Temp\ipykernel_60220\3405042881.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df2['jaundice'] = df2['jaundice'].replace({
C:\Users\nehar\AppData\Local\Temp\ipykernel_60220\3405042881.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.in

In [13]:
df2.head()

,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,age,gender,jaundice,austim,contry_of_res,used_app_before,result,Class/ASD
0,1,0,1,0,1,0,1,0,1,1,38,1,0,0,Austria,0,6.351166,0
1,0,0,0,0,0,0,0,0,0,0,47,0,0,0,India,0,2.255185,0
2,1,1,1,1,1,1,1,1,1,1,7,0,0,1,United States,0,14.851484,1
3,0,0,0,0,0,0,0,0,0,0,23,1,0,0,United States,0,2.276617,0
4,0,0,0,0,0,0,0,0,0,0,43,0,0,0,South Africa,0,-4.777286,0


In [14]:
print(df2['jaundice'].unique())

[0 1]


In [15]:
dummies = pd.get_dummies(df2['contry_of_res']).astype(int)
dummies

,Afghanistan,AmericanSamoa,Angola,Argentina,Armenia,Aruba,Australia,Austria,Azerbaijan,Bahamas,...,South Africa,Spain,Sri Lanka,Sweden,Tonga,Ukraine,United Arab Emirates,United Kingdom,United States,Viet Nam
0,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
796,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
797,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
798,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
merged = pd.concat([df2,dummies],axis='columns')
merged

,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,...,South Africa,Spain,Sri Lanka,Sweden,Tonga,Ukraine,United Arab Emirates,United Kingdom,United States,Viet Nam
0,1,0,1,0,1,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,1,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,0,1,0,0,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
796,0,1,1,0,0,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
797,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
798,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
final = merged.drop(["contry_of_res","Viet Nam"],axis='columns')
final

,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,...,Sierra Leone,South Africa,Spain,Sri Lanka,Sweden,Tonga,Ukraine,United Arab Emirates,United Kingdom,United States
0,1,0,1,0,1,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,0,1,0,0,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
796,0,1,1,0,0,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
797,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
798,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
final.shape

(800, 72)

In [19]:
X= final.drop('Class/ASD',axis='columns')
X

,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,...,Sierra Leone,South Africa,Spain,Sri Lanka,Sweden,Tonga,Ukraine,United Arab Emirates,United Kingdom,United States
0,1,0,1,0,1,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,0,1,0,0,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
796,0,1,1,0,0,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
797,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
798,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
y = final['Class/ASD']
y

0      0
1      0
2      1
3      0
4      0
      ..
795    0
796    0
797    0
798    0
799    0
Name: Class/ASD, Length: 800, dtype: int64

In [21]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X,y)
model.score(X,y)

0.45862432607604053

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=1000, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True)
}

kf = KFold(n_splits=8, shuffle=True, random_state=42)

for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=kf, scoring='accuracy')
    print(f"{name} - Mean Accuracy: {scores.mean():.4f} | Std Dev: {scores.std():.4f}")


Logistic Regression - Mean Accuracy: 0.8425 | Std Dev: 0.0435
Random Forest - Mean Accuracy: 0.8612 | Std Dev: 0.0348
SVM - Mean Accuracy: 0.8562 | Std Dev: 0.0250


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

from sklearn.tree import DecisionTreeClassifier
base_estimator = DecisionTreeClassifier()


bagging_model = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=1000, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True),
    "Bagging (DT base)": bagging_model
}

kf = KFold(n_splits=6, shuffle=True, random_state=42)

for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=kf, scoring='accuracy')
    print(f"{name} - Mean Accuracy: {scores.mean():.4f} | Std Dev: {scores.std():.4f}")


Logistic Regression - Mean Accuracy: 0.8475 | Std Dev: 0.0328
Random Forest - Mean Accuracy: 0.8637 | Std Dev: 0.0286
SVM - Mean Accuracy: 0.8538 | Std Dev: 0.0197
Bagging (DT base) - Mean Accuracy: 0.8525 | Std Dev: 0.0311


In [1]:


final_model = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42)
final_model.fit(X_scaled, y)




NameError: name 'BaggingClassifier' is not defined

In [25]:
test = pd.read_csv("test.csv")

In [26]:
test["age"] = test["age"].astype(int)
test.head()

,ID,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,...,age,gender,ethnicity,jaundice,austim,contry_of_res,used_app_before,result,age_desc,relation
0,1,1,1,0,0,1,1,0,0,1,...,15,m,White-European,yes,no,India,no,12.399055,18 and more,Self
1,2,1,0,0,0,0,0,0,1,0,...,27,m,Asian,no,no,Mexico,no,6.551598,18 and more,Self
2,3,1,1,1,0,1,1,0,1,1,...,31,m,White-European,yes,no,Egypt,no,3.180663,18 and more,Self
3,4,0,0,0,0,0,0,0,0,0,...,25,m,?,no,no,India,no,2.220766,18 and more,Self
4,5,0,0,0,1,0,0,0,0,0,...,9,m,?,no,no,Italy,no,7.252028,18 and more,Self


In [27]:
test.isnull().sum()

ID                 0
A1_Score           0
A2_Score           0
A3_Score           0
A4_Score           0
A5_Score           0
A6_Score           0
A7_Score           0
A8_Score           0
A9_Score           0
A10_Score          0
age                0
gender             0
ethnicity          0
jaundice           0
austim             0
contry_of_res      0
used_app_before    0
result             0
age_desc           0
relation           0
dtype: int64

In [28]:
for col in test.columns:
    num_cols = ["ID", "age", "result"]
    if col not in num_cols:
        print(col, test[col].unique())
        print("-"*50)

A1_Score [1 0]
--------------------------------------------------
A2_Score [1 0]
--------------------------------------------------
A3_Score [0 1]
--------------------------------------------------
A4_Score [0 1]
--------------------------------------------------
A5_Score [1 0]
--------------------------------------------------
A6_Score [1 0]
--------------------------------------------------
A7_Score [0 1]
--------------------------------------------------
A8_Score [0 1]
--------------------------------------------------
A9_Score [1 0]
--------------------------------------------------
A10_Score [1 0]
--------------------------------------------------
gender ['m' 'f']
--------------------------------------------------
ethnicity ['White-European' 'Asian' '?' 'Middle Eastern ' 'South Asian' 'Pasifika'
 'Turkish' 'Latino' 'Black' 'Others' 'Hispanic']
--------------------------------------------------
jaundice ['yes' 'no']
--------------------------------------------------
austim ['no' 'y

In [29]:
df4 = test.drop(columns=["ID", "age_desc","relation","ethnicity"])

In [30]:
print(df4['gender'].unique())
df4['gender'] = df4['gender'].replace({
    'f': 1, 'female': 1,
    'm': 0, 'male': 0
})
df4['jaundice'] = df4['jaundice'].replace({
    'yes': 1, 
    'no': 0
})
df4['austim'] = df4['austim'].replace({
    'yes': 1, 
    'no': 0
})
df4['used_app_before'] = df4['used_app_before'].replace({
    'yes': 1, 
    'no': 0
})


['m' 'f']


C:\Users\nehar\AppData\Local\Temp\ipykernel_60220\3197639789.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df4['gender'] = df4['gender'].replace({
C:\Users\nehar\AppData\Local\Temp\ipykernel_60220\3197639789.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df4['jaundice'] = df4['jaundice'].replace({
C:\Users\nehar\AppData\Local\Temp\ipykernel_60220\3197639789.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.in

In [31]:
dummies = pd.get_dummies(df4['contry_of_res']).astype(int)
dummies

,Afghanistan,Aruba,Australia,Austria,Azerbaijan,Bahamas,Bolivia,Burundi,Canada,Czech Republic,...,Philippines,Russia,Spain,Sri Lanka,Tonga,United Arab Emirates,United Kingdom,United States,Uruguay,Viet Nam
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
196,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
197,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
198,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [32]:
merged1 = pd.concat([df4,dummies],axis='columns')
merged1

,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,...,Philippines,Russia,Spain,Sri Lanka,Tonga,United Arab Emirates,United Kingdom,United States,Uruguay,Viet Nam
0,1,1,0,0,1,1,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,0,1,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,1,1,0,0,1,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
196,1,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
197,1,0,0,0,0,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
198,0,1,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,1,0,0


In [33]:
final1 = merged1.drop(["contry_of_res","Viet Nam"],axis='columns')
final1

,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,...,Pakistan,Philippines,Russia,Spain,Sri Lanka,Tonga,United Arab Emirates,United Kingdom,United States,Uruguay
0,1,1,0,0,1,1,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,0,1,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,1,1,0,0,1,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
196,1,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
197,1,0,0,0,0,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
198,0,1,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,1,0


In [34]:
final1.shape

(200, 50)

In [35]:
import pandas as pd

test = pd.read_csv("test.csv")

test["age"] = test["age"].astype(int)
df4 = test.drop(columns=["ID", "age_desc", "relation", "ethnicity"])

df4['gender'] = df4['gender'].replace({'f': 1, 'female': 1, 'm': 0, 'male': 0})
df4['jaundice'] = df4['jaundice'].replace({'yes': 1, 'no': 0})
df4['austim'] = df4['austim'].replace({'yes': 1, 'no': 0})
df4['used_app_before'] = df4['used_app_before'].replace({'yes': 1, 'no': 0})

dummies = pd.get_dummies(df4['contry_of_res']).astype(int)
merged1 = pd.concat([df4, dummies], axis='columns')
final1 = merged1.drop(["contry_of_res", "Viet Nam"], axis='columns')  # Drop reference country

missing_cols = set(X.columns) - set(final1.columns)
for col in missing_cols:
    final1[col] = 0  

final1 = final1[X.columns]  

X_test_scaled = scaler.transform(final1)

y_test_pred = final_model.predict(X_test_scaled)


submission = pd.DataFrame({
    'ID': test['ID'],         
    'Class/ASD': y_test_pred   
})

submission.to_csv("submission.csv", index=False)



C:\Users\nehar\AppData\Local\Temp\ipykernel_60220\3621122555.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df4['gender'] = df4['gender'].replace({'f': 1, 'female': 1, 'm': 0, 'male': 0})
C:\Users\nehar\AppData\Local\Temp\ipykernel_60220\3621122555.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df4['jaundice'] = df4['jaundice'].replace({'yes': 1, 'no': 0})
C:\Users\nehar\AppData\Local\Temp\ipykernel_60220\3621122555.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future vers